In [2]:
!pip install -U segmentation-models-pytorch albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.4/369.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninst

In [2]:
import os
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transforms=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transforms = transforms

        self.image_filenames = sorted([
            f for f in os.listdir(image_dir) if f.endswith(".jpg")
        ])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        image_name = self.image_filenames[idx]
        image_path = os.path.join(self.image_dir, image_name)
        mask_path = os.path.join(self.mask_dir, image_name.replace(".jpg", ".png"))
    
        image = np.array(Image.open(image_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))
    
        # Resize mask to match image if necessary
        if image.shape[:2] != mask.shape:
            mask = np.array(Image.fromarray(mask).resize((image.shape[1], image.shape[0]), resample=Image.NEAREST))
    
        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"].long()
    
        return image, mask


In [4]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])

train_dataset = SegmentationDataset(
    image_dir="/kaggle/input/visual-redactions/images/train",
    mask_dir="/kaggle/input/visual-masks/masks/train",
    transforms=transform
)

val_dataset = SegmentationDataset(
    image_dir="/kaggle/input/visual-redactions/images/val",
    mask_dir="/kaggle/input/visual-masks/masks/val",
    transforms=transform
)

print(f"✅ Loaded {len(train_dataset)} train / {len(val_dataset)} val samples")


✅ Loaded 3846 train / 1604 val samples


In [21]:
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = F.softmax(preds, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0, 3, 1, 2).float()

        intersection = (preds * targets_one_hot).sum(dim=(2, 3))
        union = preds.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))

        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


In [22]:
ce_loss = nn.CrossEntropyLoss(weight=weights.to(device))
dice_loss = DiceLoss()

def hybrid_loss(preds, targets):
    return ce_loss(preds, targets) + dice_loss(preds, targets)


In [23]:
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch

NUM_CLASSES = 10

model = smp.DeepLabV3Plus(
    encoder_name="resnet101",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
)


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("✅ Model ready on", device)


✅ Model ready on cuda


In [24]:
from torch.utils.data import DataLoader
from tqdm import tqdm

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = hybrid_loss(outputs, masks)


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == masks).sum().item()
        total += torch.numel(masks)

        loop.set_postfix(loss=loss.item(), acc=100. * correct / total)

    print(f"✅ Epoch {epoch+1} complete — Loss: {total_loss:.4f} | Acc: {100. * correct / total:.2f}%")


Epoch 1/5: 100%|██████████| 962/962 [08:30<00:00,  1.88it/s, acc=65.3, loss=1.42]


✅ Epoch 1 complete — Loss: 1991.6971 | Acc: 65.27%


Epoch 2/5: 100%|██████████| 962/962 [08:29<00:00,  1.89it/s, acc=73.5, loss=2.29] 


✅ Epoch 2 complete — Loss: 1707.0698 | Acc: 73.49%


Epoch 3/5: 100%|██████████| 962/962 [08:28<00:00,  1.89it/s, acc=74.9, loss=5.67] 


✅ Epoch 3 complete — Loss: 1590.1335 | Acc: 74.91%


Epoch 4/5: 100%|██████████| 962/962 [08:29<00:00,  1.89it/s, acc=76.2, loss=3.57] 


✅ Epoch 4 complete — Loss: 1517.4850 | Acc: 76.24%


Epoch 5/5: 100%|██████████| 962/962 [08:28<00:00,  1.89it/s, acc=77.5, loss=1.25] 

✅ Epoch 5 complete — Loss: 1452.9582 | Acc: 77.47%


In [25]:
torch.save(model.state_dict(), "deeplabv3plus_privacy_hybrid.pth")


In [26]:
model.eval()
val_inter = np.zeros(NUM_CLASSES)
val_union = np.zeros(NUM_CLASSES)
CLASS_NAMES = [
    "background", "face", "license_plate", "person_body", "nudity",
    "handwriting", "disability", "medicine", "fingerprint", "signature"
]
with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="🔍 Validating"):
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        preds = outputs.argmax(1)

        for cls in range(NUM_CLASSES):
            pred_inds = (preds == cls)
            target_inds = (masks == cls)

            intersection = (pred_inds & target_inds).sum().item()
            union = (pred_inds | target_inds).sum().item()

            val_inter[cls] += intersection
            val_union[cls] += union

# Calculate IoUs
val_class_ious = val_inter / np.maximum(val_union, 1)
val_miou = val_class_ious.mean()

print(f"✅ Validation mIoU: {val_miou:.4f}")
print("📊 Per-Class IoUs:")
for i, iou in enumerate(val_class_ious):
    print(f"  {CLASS_NAMES[i]:<15}: {iou:.4f}")


🔍 Validating: 100%|██████████| 401/401 [01:46<00:00,  3.76it/s]

✅ Validation mIoU: 0.2919
📊 Per-Class IoUs:
  background     : 0.6950
  face           : 0.6260
  license_plate  : 0.0735
  person_body    : 0.5657
  nudity         : 0.1338
  handwriting    : 0.4284
  disability     : 0.1392
  medicine       : 0.0293
  fingerprint    : 0.1550
  signature      : 0.0730


In [12]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from collections import Counter

def compute_class_weights(mask_dir, num_classes=10):
    pixel_counts = Counter()

    for fname in os.listdir(mask_dir):
        if fname.endswith(".png"):
            mask_path = os.path.join(mask_dir, fname)
            mask = np.array(Image.open(mask_path))
            unique, counts = np.unique(mask, return_counts=True)
            pixel_counts.update(dict(zip(unique, counts)))

    total_pixels = sum(pixel_counts.values())
    weights = []

    for cls in range(num_classes):
        count = pixel_counts.get(cls, 1)
        weight = total_pixels / (num_classes * count)
        weights.append(weight)

    weights = torch.tensor(weights, dtype=torch.float32)
    weights /= weights.sum()  # normalize

    return weights
